In [1]:
import io
import os
import sys
import traceback
from typing import Optional

# Importaciones necesarias para procesamiento y visualización de imágenes.
try:
    from PIL import Image
except Exception:
    print("La biblioteca Pillow no está instalada. Por favor, ejecuta: pip install pillow")
    raise

import numpy as np
import matplotlib.pyplot as plt

# Intentar importar OpenCV; si no está, instalar automáticamente (si es posible).
try:
    import cv2
except Exception:
    print("OpenCV no encontrado. Instalando opencv-python-headless...")
    try:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "opencv-python-headless"])
        import cv2
    except Exception:
        print("No se pudo instalar OpenCV. Instala manualmente e intenta luego.")
        raise

# Importar imageio para generar GIFs animados
try:
    import imageio
except Exception:
    print("imageio no instalado. Intentando instalar...")
    try:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "imageio"])
        import imageio
    except Exception:
        print("No se pudo instalar imageio. Hazlo manualmente e intenta luego.")
        raise

# Intentar importar herramientas de Colab para carga y descarga de archivos.
IS_COLAB = False
try:
    from google.colab import files as colab_files
    IS_COLAB = True
except Exception:
    IS_COLAB = False

# Intentar soporte de display para notebooks
try:
    from IPython.display import display, Image as IPyImage
except Exception:
    def display(x):
        print("Display no disponible fuera de un notebook.")
    IPyImage = None


# FUNCIONES AUXILIARES

def upload_image_interactive() -> Optional[np.ndarray]:
    """
    Carga una imagen a color (RGB) desde archivo local, URL o Colab.
    Devuelve la imagen como array NumPy RGB (uint8).
    """
    if IS_COLAB:
        uploaded = colab_files.upload()
        if not uploaded:
            print("No se subió ningún archivo.")
            return None
        fname = next(iter(uploaded))
        img_pil = Image.open(io.BytesIO(uploaded[fname])).convert("RGB")
        img = np.array(img_pil)
        print(f"Cargada imagen desde Colab: {fname} — dimensión: {img.shape}")
        return img
    else:
        print("Carga de imagen. Opción:")
        choice = input("¿Usar (1) archivo local o (2) URL? [1/2]: ").strip()
        if choice == "1":
            path = input("Ruta local de la imagen: ").strip()
            if not os.path.exists(path):
                print("Archivo no encontrado.")
                return None
            img_pil = Image.open(path).convert("RGB")
            img = np.array(img_pil)
            return img
        elif choice == "2":
            url = input("URL de la imagen: ").strip()
            try:
                from urllib.request import urlopen
                resp = urlopen(url)
                img_pil = Image.open(io.BytesIO(resp.read())).convert("RGB")
                img = np.array(img_pil)
                return img
            except Exception as e:
                print(f"No se pudo descargar la imagen: {e}")
                return None
        else:
            print("Opción incorrecta.")
            return None

def show_image(img: np.ndarray, title: str = ""):
    """
    Muestra una imagen RGB usando matplotlib.
    """
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")
    plt.show()

def ensure_uint8(arr: np.ndarray) -> np.ndarray:
    """
    Convierte array al rango [0,255] y tipo uint8.
    """
    return np.clip(arr, 0, 255).astype(np.uint8)

def to_rgb_from_gray(gray: np.ndarray) -> np.ndarray:
    """
    Convierte imagen en escala de grises a RGB replicando el canal para visualización.
    """
    if gray.ndim == 2:
        return np.stack([gray]*3, axis=-1)
    return gray


# PROCESAMIENTO PRINCIPAL / PIPELINE

def process_image_pipeline(img: np.ndarray, output_gif_path: str = "gifs/animal_proceso.gif"):
    """
    Ejecuta el procesamiento solicitado: filtros, canales, morfología y crea GIF.
    Devuelve path del GIF creado.
    """
    print("Procesamiento de imagen en marcha...")
    img = ensure_uint8(img)
    show_image(img, "Imagen original (RGB)")

    # ----------- Filtros básicos -----------
    print("Aplicando filtros: suavizado y realce.")
    blur = cv2.GaussianBlur(img, ksize=(7,7), sigmaX=0)
    show_image(blur, "Filtro de Suavizado (Desenfoque)")

    print("El suavizado reduce el ruido y detalles finos, dejando la imagen menos definida.")
    kernel_sharpen = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
    sharpen = cv2.filter2D(img, -1, kernel_sharpen)
    sharpen = ensure_uint8(sharpen)
    show_image(sharpen, "Filtro de Realce de Bordes")

    print("El filtro de realce aumenta el contraste en las aristas y destaca los bordes, pero puede amplificar el ruido.")

    # ----------- Visualización de canales ----------
    print("Separando canales de color (R, G, B).")
    R, G, B = img[:,:,0], img[:,:,1], img[:,:,2]
    fig, axes = plt.subplots(1,3, figsize=(18,6))
    axes[0].imshow(R, cmap="gray"); axes[0].set_title("Canal Rojo (R)"); axes[0].axis("off")
    axes[1].imshow(G, cmap="gray"); axes[1].set_title("Canal Verde (G)"); axes[1].axis("off")
    axes[2].imshow(B, cmap="gray"); axes[2].set_title("Canal Azul (B)"); axes[2].axis("off")
    plt.show()

    print("En el canal R, áreas rojizas o tonos cálidos aparecen más claras. El canal G suele resaltar vegetación y texturas. El canal B muestra mayor claridad en zonas frías u oscuras, como sombras azuladas.")

    # ---------- Operaciones morfológicas ----------
    print("Operaciones morfológicas sobre la imagen binarizada (Otsu).")
    img_gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, binary = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7,7))
    opening = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    closing = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    fig, axes = plt.subplots(1,4, figsize=(22,6))
    axes[0].imshow(img_gray, cmap="gray"); axes[0].set_title("Escala de grises"); axes[0].axis("off")
    axes[1].imshow(binary, cmap="gray"); axes[1].set_title("Binarizada (Otsu)"); axes[1].axis("off")
    axes[2].imshow(opening, cmap="gray"); axes[2].set_title("Opening"); axes[2].axis("off")
    axes[3].imshow(closing, cmap="gray"); axes[3].set_title("Closing"); axes[3].axis("off")
    plt.show()

    print("La apertura (opening) elimina ruido blanco pequeño y puede adelgazar estructuras. El cierre (closing) rellena huecos pequeños y conecta objetos fragmentados.")

    # ----------- Generación de GIF animado ----------
    print("Generando GIF con pasos secuenciales de procesamiento.")
    frames = []
    def pil_from_arr(arr):
        arr = np.asarray(arr)
        if arr.ndim == 2:
            arr_rgb = np.stack([arr]*3, axis=-1)
        else:
            arr_rgb = arr
        arr_rgb = ensure_uint8(arr_rgb)
        return Image.fromarray(arr_rgb)

    frames.append(pil_from_arr(img))                          # original
    frames.append(pil_from_arr(blur))                         # suavizado
    frames.append(pil_from_arr(sharpen))                      # realce
    frames.append(pil_from_arr(binary))                       # binarizada
    frames.append(pil_from_arr(opening))                      # opening
    frames.append(pil_from_arr(closing))                      # closing

    # Crear carpeta de gifs/ si no existe
    gif_dir = os.path.dirname(output_gif_path)
    if gif_dir and not os.path.exists(gif_dir):
        os.makedirs(gif_dir, exist_ok=True)

    try:
        imageio.mimsave(output_gif_path, [np.array(f) for f in frames], duration=0.7)
        print(f"GIF secuencia guardado en: {output_gif_path}")
    except Exception:
        print("Error generando GIF con imageio. Intentando con PIL...")
        try:
            frames[0].save(output_gif_path, save_all=True, append_images=frames[1:], duration=700, loop=0)
            print(f"GIF guardado con PIL en: {output_gif_path}")
        except Exception:
            print("No se pudo crear el GIF.")

    # Display en notebook si corresponde
    if IPyImage and os.path.exists(output_gif_path):
        try:
            display(IPyImage(filename=output_gif_path))
        except Exception:
            print("No es posible mostrar el GIF en este entorno.")
    elif os.path.exists(output_gif_path):
        print(f"Descarga el GIF desde el sistema de archivos: ./{output_gif_path}")

    return output_gif_path


# ENTRADA PRINCIPAL

def main():
    print("=== Procesamiento de Imagen de Animal en Peligro de Extinción ===")
    try:
        img = upload_image_interactive()
        if img is None:
            print("No se cargó imagen. Usando imagen de tigre desde Wikimedia como ejemplo.")
            try:
                from urllib.request import urlopen
                url = "https://upload.wikimedia.org/wikipedia/commons/6/6d/Amur_tiger_in_Sikhote-Alin%2C_Russia.jpg"
                resp = urlopen(url)
                img_pil = Image.open(io.BytesIO(resp.read())).convert("RGB")
                img = np.array(img_pil)
                print("Imagen de ejemplo descargada correctamente.")
            except Exception:
                print("Error al descargar imagen de ejemplo. Saliendo.")
                traceback.print_exc()
                return

        gif_path = process_image_pipeline(img, output_gif_path="gifs/animal_proceso.gif")

        if IS_COLAB and os.path.exists(gif_path):
            try:
                from google.colab import files
                files.download(gif_path)
            except Exception:
                print("No se pudo ofrecer descarga automática en Colab.")
        print("Procesamiento completado correctamente.")

    except Exception as e:
        print(f"Error durante la ejecución: {e}")
        traceback.print_exc()

if __name__ == "__main__":
    main()

Output hidden; open in https://colab.research.google.com to view.